# Advanced Forecasting Models

## 1. Setup — connexion DB, imports, project_root

In [2]:
import sys
from dotenv import load_dotenv

load_dotenv(r"c:\Users\admin\demand-forecasting\.env")

sys.path.append(r"c:\Users\admin\demand-forecasting")

In [3]:
import os

project_root = r"c:\Users\admin\demand-forecasting"
output_dir = os.path.join(project_root, "data", "processed")
os.makedirs(output_dir, exist_ok=True)


In [4]:
from etl.load import get_engine
import pandas as pd
import numpy as np
import lightgbm as lgb

engine = get_engine()

In [5]:
import glob

output_dir = os.path.join(project_root, "data", "processed")

dataframes = {}

for filepath in glob.glob(os.path.join(output_dir, "*.parquet")):
    name = os.path.splitext(os.path.basename(filepath))[0]
    dataframes[name] = pd.read_parquet(filepath)
    print(f"{name}: {dataframes[name].shape}")

full = dataframes['full']
train_fe = dataframes['train_fe']
test_fe = dataframes['test_fe']

df_bornes: (30490, 5)
df_left: (58327370, 4)
df_price_cat: (58327370, 6)
full: (58327370, 13)
test: (1554990, 5)
test_fe: (1554990, 13)
train: (56772380, 5)
train_fe: (56558950, 13)


## 2. Rappel baseline M4 — construction des prédictions naïves

In [4]:
query_last_sale_date = '''
    SELECT MAX(d.la_date)
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
'''
last_sale_date = pd.read_sql(query_last_sale_date, engine).iloc[0, 0]
print(last_sale_date)

cutoff_date = last_sale_date - pd.Timedelta(days=50)
print(cutoff_date)

2016-04-24
2016-03-05


In [22]:
query_train = f'''
    SELECT f.item_id, f.store_id, f.quantite, d.la_date
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    WHERE d.la_date < '{cutoff_date}'
'''

query_test = f'''
    SELECT f.item_id, f.store_id, f.quantite, d.la_date
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    WHERE d.la_date >= '{cutoff_date}'
'''

train = pd.read_sql(query_train, engine)
test = pd.read_sql(query_test, engine)

print(train.shape, test.shape)

(56772380, 4) (1554990, 4)


In [23]:
# Dernier jour de chaque série (item+store) dans train, par jour de semaine
train['la_date'] = pd.to_datetime(train['la_date'])
train['jour_semaine'] = train['la_date'].dt.dayofweek  # 0=lundi, 6=dimanche

# Pour chaque item+store+jour_de_semaine, la dernière valeur connue dans train
last_known = (
    train
    .sort_values('la_date')
    .groupby(['item_id', 'store_id', 'jour_semaine'])
    .last()
    .reset_index()[['item_id', 'store_id', 'jour_semaine', 'quantite']]
    .rename(columns={'quantite': 'prediction_naive_saisonnier'})
)

test['la_date'] = pd.to_datetime(test['la_date'])
test['jour_semaine'] = test['la_date'].dt.dayofweek

test_with_pred = test.merge(
    last_known,
    on=['item_id', 'store_id', 'jour_semaine'],
    how='left'
)

print(test_with_pred.shape)
print(test_with_pred.isna().sum())
print(test_with_pred.head())

(1554990, 6)
item_id                        0
store_id                       0
quantite                       0
la_date                        0
jour_semaine                   0
prediction_naive_saisonnier    0
dtype: int64
   item_id  store_id  quantite    la_date  jour_semaine  \
0        1         1         0 2016-03-05             5   
1        2         1         0 2016-03-05             5   
2        3         1         6 2016-03-05             5   
3        4         1         1 2016-03-05             5   
4        5         1         1 2016-03-05             5   

   prediction_naive_saisonnier  
0                            4  
1                            0  
2                            0  
3                            0  
4                            1  


In [7]:
# Dernière valeur connue (dernier jour) pour chaque item+store dans train
last_value = (
    train
    .sort_values('la_date')
    .groupby(['item_id', 'store_id'])
    .last()
    .reset_index()[['item_id', 'store_id', 'quantite']]
    .rename(columns={'quantite': 'prediction_naive_simple'})
)

test_with_pred = test_with_pred.merge(
    last_value,
    on=['item_id', 'store_id'],
    how='left'
)

print(test_with_pred[['quantite', 'prediction_naive_saisonnier', 'prediction_naive_simple']].head())
print(test_with_pred.isna().sum())

   quantite  prediction_naive_saisonnier  prediction_naive_simple
0         0                            4                        1
1         0                            0                        0
2         6                            0                        0
3         1                            0                        0
4         1                            1                        0
item_id                        0
store_id                       0
quantite                       0
la_date                        0
jour_semaine                   0
prediction_naive_saisonnier    0
prediction_naive_simple        0
dtype: int64


## 3. Sauvegarde train / test bruts (checkpoint Parquet)

In [24]:
train.to_parquet(os.path.join(output_dir, "train.parquet"), index=False)
test.to_parquet(os.path.join(output_dir, "test.parquet"), index=False)

## 4. Évaluation de la baseline naïve (WAPE de référence M4)

In [10]:
def wape(y_true, y_pred):
    return (y_true - y_pred).abs().sum() / y_true.sum()

In [ ]:
wape_naive_simple = wape(test_with_pred['quantite'], test_with_pred['prediction_naive_simple'])
wape_naive_saisonnier = wape(test_with_pred['quantite'], test_with_pred['prediction_naive_saisonnier'])

print(f"WAPE naïf simple : {wape_naive_simple:.4f}")
print(f"WAPE naïf saisonnier : {wape_naive_saisonnier:.4f}")

WAPE naïf simple : 0.9042
WAPE naïf saisonnier : 0.9160


## 5. Feature Engineering — Lags (lag_1, lag_7)

*Calculés sur `train`+`test` concaténés pour éviter la fuite temporelle, puis re-séparés.*

In [11]:
full = pd.concat([train,test]).sort_values(['item_id', 'store_id', 'la_date'])

full['lag_1'] = full.groupby(['item_id', 'store_id'])['quantite'].shift(1)
full['lag_7'] = full.groupby(['item_id', 'store_id'])['quantite'].shift(7)

train_fe = full[full['la_date'] < cutoff_date].copy()
test_fe = full[full['la_date'] >= cutoff_date].copy()

print(train_fe[['lag_1', 'lag_7']].isna().sum())
print(test_fe[['lag_1', 'lag_7']].isna().sum())

lag_1     30490
lag_7    213430
dtype: int64
lag_1    0
lag_7    0
dtype: int64


In [12]:
test_fe.to_parquet(os.path.join(output_dir, "test_fe.parquet"), index=False)

In [13]:
train_fe = train_fe.dropna(subset=['lag_1', 'lag_7'])
print(train_fe.shape)

train_fe.to_parquet(os.path.join(output_dir, "train_fe.parquet"), index=False)

(56558950, 6)


## 6. Feature Engineering — Rolling mean (rolling_mean_7)

In [14]:
full['rolling_mean_7'] = (
    full.groupby(['item_id', 'store_id'])['quantite']
    .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

print(full['rolling_mean_7'].isna().sum())

full.to_parquet(os.path.join(output_dir, "full.parquet"), index=False)

213430


## 7. Investigation — jointure prix et catégorie

*Diagnostic de l'écart de ~12M lignes entre `full` et la jointure prix (INNER JOIN → LEFT JOIN).*

In [5]:
query_price_join = '''
    SELECT 
        f.item_id,
        f.store_id,
        d.la_date,
        p.price,
        i.categorie
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    JOIN item i ON f.item_id = i.id
    LEFT JOIN price p 
        ON f.item_id = p.item_id 
        AND f.store_id = p.store_id
        AND d.la_date >= p.valid_from
        AND (d.la_date < p.valid_to OR p.valid_to IS NULL)
'''

df_price_cat = pd.read_sql(query_price_join, engine)
df_price_cat['la_date'] = pd.to_datetime(df_price_cat['la_date'])
df_price_cat['prix_connu'] = df_price_cat['price'].notna().astype(int)

print(df_price_cat.shape)
print(df_price_cat['prix_connu'].value_counts())

(58327370, 6)
prix_connu
1    46027957
0    12299413
Name: count, dtype: int64


In [6]:
df_price_cat.to_parquet(os.path.join(output_dir, "df_price_cat.parquet"), index=False)

In [17]:
query_left_join = '''
    SELECT 
        f.item_id,
        f.store_id,
        d.la_date,
        p.price
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    LEFT JOIN price p 
        ON f.item_id = p.item_id 
        AND f.store_id = p.store_id
        AND d.la_date >= p.valid_from
        AND (d.la_date < p.valid_to OR p.valid_to IS NULL)
'''

df_left = pd.read_sql(query_left_join, engine)
print(df_left.shape)
print(df_left['price'].isna().sum())

(58327370, 4)
12299413


In [18]:
df_left.to_parquet(os.path.join(output_dir, "df_left.parquet"), index=False)

In [19]:
items_in_sales = set(pd.read_sql("SELECT DISTINCT item_id FROM fact_sales", engine)['item_id'])
items_in_price = set(pd.read_sql("SELECT DISTINCT item_id FROM price", engine)['item_id'])

items_sans_prix = items_in_sales - items_in_price
print(f"Items dans fact_sales : {len(items_in_sales)}")
print(f"Items dans price : {len(items_in_price)}")
print(f"Items SANS aucune ligne de prix : {len(items_sans_prix)}")

Items dans fact_sales : 3049
Items dans price : 3049
Items SANS aucune ligne de prix : 0


In [20]:
query_bornes = '''
    SELECT 
        f.item_id,
        f.store_id,
        MIN(d.la_date) AS premiere_vente,
        MIN(p.valid_from) AS premier_prix_connu
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    LEFT JOIN price p ON f.item_id = p.item_id AND f.store_id = p.store_id
    GROUP BY f.item_id, f.store_id
'''

df_bornes = pd.read_sql(query_bornes, engine)
df_bornes['ecart_jours'] = (pd.to_datetime(df_bornes['premier_prix_connu']) - pd.to_datetime(df_bornes['premiere_vente'])).dt.days

print(df_bornes['ecart_jours'].describe())
print((df_bornes['ecart_jours'] > 0).sum(), "combinaisons item+store où le prix arrive APRÈS la première vente")

count    30490.000000
mean       403.391702
std        476.005658
min          0.000000
25%          0.000000
50%        154.000000
75%        763.000000
max       1841.000000
Name: ecart_jours, dtype: float64
19558 combinaisons item+store où le prix arrive APRÈS la première vente


In [21]:
df_bornes.to_parquet(os.path.join(output_dir, "df_bornes.parquet"), index=False)

In [26]:
print(df_left.shape)
df_bornes['ecart_jours'].describe()

(58327370, 4)


count    30490.000000
mean       403.391702
std        476.005658
min          0.000000
25%          0.000000
50%        154.000000
75%        763.000000
max       1841.000000
Name: ecart_jours, dtype: float64

## 8. Reconstruction de `full` enrichi

*Rechargement depuis Parquet + merge de la version corrigée de `df_price_cat` (LEFT JOIN, flag `prix_connu`).*

In [5]:
full = pd.read_parquet(os.path.join(output_dir, "full.parquet"))
df_price_cat = pd.read_parquet(os.path.join(output_dir, "df_price_cat.parquet"))

In [9]:
full['la_date'] = pd.to_datetime(full['la_date'])

full = full.merge(
    df_price_cat[['item_id', 'store_id', 'la_date', 'price', 'categorie', 'prix_connu']],
    on=['item_id', 'store_id', 'la_date'],
    how='left'
)

print(full.shape)
print(full[['price', 'categorie', 'prix_connu']].isna().sum())

(58327370, 10)
price         12299413
categorie            0
prix_connu           0
dtype: int64


## 9. Ajout des features calendaires (jour de semaine, weekend, jour férié)

In [10]:
query_calendar = '''
    SELECT la_date, jour_de_semaine, is_weekend, is_holiday
    FROM dim_date
'''
df_calendar = pd.read_sql(query_calendar, engine)
df_calendar['la_date'] = pd.to_datetime(df_calendar['la_date'])

full = full.merge(df_calendar, on='la_date', how='left')

print(full.shape)
print(full[['jour_de_semaine', 'is_weekend', 'is_holiday']].isna().sum())

(58327370, 13)
jour_de_semaine    0
is_weekend         0
is_holiday         0
dtype: int64


In [ ]:
full['jour_semaine_num'] = full['la_date'].dt.dayofweek  # 0=lundi, 6=dimanche
full = full.drop(columns=['jour_de_semaine'])

KeyError: "['jour_de_semaine'] not found in axis"

In [13]:
full.to_parquet(os.path.join(output_dir, "full.parquet"), index=False)

## 10. Split final train_fe / test_fe (cutoff 2016-03-05)

In [15]:
cutoff_date = '2016-03-05'

train_fe = full[full['la_date'] < cutoff_date].copy()
test_fe = full[full['la_date'] >= cutoff_date].copy()

print(train_fe.shape)
print(test_fe.shape)
print(train_fe[['lag_1', 'lag_7', 'rolling_mean_7']].isna().sum())
print(test_fe[['lag_1', 'lag_7', 'rolling_mean_7']].isna().sum())

train_fe = train_fe.dropna(subset=['lag_1', 'lag_7'])
print(train_fe.shape)

(56772380, 13)
(1554990, 13)
lag_1              30490
lag_7             213430
rolling_mean_7    213430
dtype: int64
lag_1             0
lag_7             0
rolling_mean_7    0
dtype: int64
(56558950, 13)


In [16]:
train_fe.to_parquet(os.path.join(output_dir, "train_fe.parquet"), index=False)
test_fe.to_parquet(os.path.join(output_dir, "test_fe.parquet"), index=False)

## 11. Préparation des features pour LightGBM (typage catégoriel)

In [6]:
features = ['lag_1', 'lag_7', 'rolling_mean_7', 'price', 'prix_connu', 
            'categorie', 'jour_semaine_num', 'is_weekend', 'is_holiday']

X_train = train_fe[features].copy()
y_train = train_fe['quantite']

X_test = test_fe[features].copy()
y_test = test_fe['quantite']

In [7]:
cat_features = ['categorie', 'jour_semaine_num', 'is_weekend', 'is_holiday']

for col in cat_features:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(X_train.dtypes)

lag_1                float64
lag_7                float64
rolling_mean_7       float64
price                float64
prix_connu             int64
categorie           category
jour_semaine_num    category
is_weekend          category
is_holiday          category
dtype: object


## 12. Entraînement du modèle LightGBM (baseline)

In [8]:
model = lgb.LGBMRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train, categorical_feature=cat_features)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.743988 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9
[LightGBM] [Info] Start training from score 1.119872


,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [11]:
y_pred = model.predict(X_test)
y_pred = y_pred.clip(min=0)

wape_lgbm = wape(y_test, pd.Series(y_pred, index=y_test.index))
print(f"WAPE LightGBM : {wape_lgbm:.4f}")

WAPE LightGBM : 0.7228


## 13. Analyse des résultats — feature importance, WAPE par catégorie, intermittence

In [ ]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

            feature  importance
2    rolling_mean_7         737
0             lag_1         667
3             price         567
1             lag_7         425
6  jour_semaine_num         322
5         categorie         114
7        is_weekend         108
8        is_holiday          60
4        prix_connu           0


In [2]:
test_fe['y_pred'] = y_pred
test_fe['erreur_abs'] = (test_fe['quantite'] - test_fe['y_pred']).abs()

wape_par_categorie = test_fe.groupby('categorie').apply(
    lambda g: wape(g['quantite'], g['y_pred'])
)
print(wape_par_categorie)


NameError: name 'y_pred' is not defined

In [3]:
# Volume de lignes par catégorie
print(train_fe['categorie'].value_counts())

# Proportion de zéros par catégorie (proxy d'intermittence)
for cat in test_fe['categorie'].unique():
    subset = test_fe[test_fe['categorie'] == cat]
    pct_zero = (subset['quantite'] == 0).mean()
    print(f"{cat}: {pct_zero:.1%} de jours à zéro vente")

NameError: name 'train_fe' is not defined

## 14. Test de segmentation par catégorie (3 modèles séparés)

*Résultat : pas de gain significatif — confirme l'intérêt du cross-learning global.*

In [4]:
wape_results = {}

for cat in train_fe['categorie'].unique():
    X_train_cat = X_train[train_fe['categorie'] == cat]
    y_train_cat = y_train[train_fe['categorie'] == cat]
    X_test_cat = X_test[test_fe['categorie'] == cat]
    y_test_cat = y_test[test_fe['categorie'] == cat]

    model_cat = lgb.LGBMRegressor(n_estimators=100, random_state=42)
    model_cat.fit(X_train_cat, y_train_cat, categorical_feature=[c for c in cat_features if c != 'categorie'])

    y_pred_cat = model_cat.predict(X_test_cat).clip(min=0)
    wape_cat = wape(y_test_cat, pd.Series(y_pred_cat, index=y_test_cat.index))
    wape_results[cat] = wape_cat
    print(f"{cat}: WAPE = {wape_cat:.4f}")

print(wape_results)

NameError: name 'train_fe' is not defined

## 15. Sauvegarde du modèle final retenu

In [ ]:
import joblib

model_dir = os.path.join(project_root, "models")
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, os.path.join(model_dir, "lgbm_baseline_m5.pkl"))


In [8]:
features = ['lag_1', 'lag_7', 'rolling_mean_7', 'price', 'prix_connu', 
            'categorie', 'jour_semaine_num', 'is_weekend', 'is_holiday']
cat_features = ['categorie', 'jour_semaine_num', 'is_weekend', 'is_holiday']

test_windows = [
    ('2016-01-15', '2016-03-05'),
    ('2016-02-01', '2016-03-22'),
    ('2016-03-05', '2016-04-24'),
]

results_backtest = []

for train_end, test_end in test_windows:
    train_bt = full[full['la_date'] < train_end].dropna(subset=['lag_1', 'lag_7'])
    test_bt = full[(full['la_date'] >= train_end) & (full['la_date'] < test_end)]

    X_train_bt = train_bt[features].copy()
    y_train_bt = train_bt['quantite']
    X_test_bt = test_bt[features].copy()
    y_test_bt = test_bt['quantite']

    for col in cat_features:
        X_train_bt[col] = X_train_bt[col].astype('category')
        X_test_bt[col] = X_test_bt[col].astype('category')

    model_bt = lgb.LGBMRegressor(n_estimators=100, random_state=42)
    model_bt.fit(X_train_bt, y_train_bt, categorical_feature=cat_features)

    y_pred_bt = model_bt.predict(X_test_bt).clip(min=0)
    wape_bt = wape(y_test_bt, pd.Series(y_pred_bt, index=y_test_bt.index))

    results_backtest.append({
        'train_end': train_end,
        'test_end': test_end,
        'wape': wape_bt,
        'n_train': len(train_bt),
        'n_test': len(test_bt)
    })

    print(f"Train < {train_end} | Test [{train_end}, {test_end}) | WAPE = {wape_bt:.4f}")

df_backtest = pd.DataFrame(results_backtest)
print(df_backtest)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.738888 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 707
[LightGBM] [Info] Number of data points in the train set: 55034450, number of used features: 9
[LightGBM] [Info] Start training from score 1.113803
Train < 2016-01-15 | Test [2016-01-15, 2016-03-05) | WAPE = 0.7305
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.775799 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 708
[LightGBM] [Info] Number of data points in the train set: 55552780, number of used features: 9
[LightGBM] [Info] Start training from score 1.115518
Train < 2016-02-01 | Test [2016-02-01, 2016-03-22) | WAPE = 0.7258
[LightGBM] [Info] Auto-choosing row-wise multi-threa

In [15]:
def mase(y_true, y_pred, y_train_naive_errors):
    mae = (y_true - y_pred).abs().mean()
    scale = y_train_naive_errors.mean()
    return mae / scale

In [16]:
naive_scale = (
    train_fe.groupby(['item_id', 'store_id'])
    .apply(lambda g: (g['quantite'] - g['lag_1']).abs().mean())
    .rename('naive_scale')
)

print(naive_scale.describe())
print((naive_scale == 0).sum(), "séries avec un scale de 0")

count    30490.000000
mean         0.918320
std          1.282072
min          0.001078
25%          0.267925
50%          0.552561
75%          1.066307
max         40.981132
Name: naive_scale, dtype: float64
0 séries avec un scale de 0


In [17]:
test_fe['y_pred'] = y_pred  

mae_model = (
    test_fe.groupby(['item_id', 'store_id'])
    .apply(lambda g: (g['quantite'] - g['y_pred']).abs().mean())
    .rename('mae_model')
)

print(mae_model.describe())

count    30490.000000
mean         1.007139
std          1.102808
min          0.036017
25%          0.442225
50%          0.720799
75%          1.169369
max         26.240212
Name: mae_model, dtype: float64


In [18]:
df_mase = pd.concat([mae_model, naive_scale], axis=1)
df_mase['mase'] = df_mase['mae_model'] / df_mase['naive_scale']

print(df_mase['mase'].describe())
print(f"MASE médian : {df_mase['mase'].median():.4f}")
print(f"% de séries avec MASE < 1 (mieux que naïf) : {(df_mase['mase'] < 1).mean():.1%}")

count    30490.000000
mean         1.877441
std          5.470555
min          0.018003
25%          0.860924
50%          1.242943
75%          1.990277
max        678.556038
Name: mase, dtype: float64
MASE médian : 1.2429
% de séries avec MASE < 1 (mieux que naïf) : 35.2%


In [19]:
# Récupérer la catégorie par item_id (fixe, donc un seul mapping suffit)
item_categorie = train_fe[['item_id', 'categorie']].drop_duplicates().set_index('item_id')

df_mase_cat = df_mase.reset_index().merge(item_categorie, on='item_id', how='left')

print(df_mase_cat.groupby('categorie')['mase'].median())
print(df_mase_cat.groupby('categorie').apply(lambda g: (g['mase'] < 1).mean()))

categorie
FOODS        1.189667
HOBBIES      1.345096
HOUSEHOLD    1.269786
Name: mase, dtype: float64
categorie
FOODS        0.385177
HOBBIES      0.307434
HOUSEHOLD    0.329226
dtype: float64


In [20]:
df_mase['volume_quartile'] = pd.qcut(df_mase['naive_scale'], 4, labels=['Q1 (faible)', 'Q2', 'Q3', 'Q4 (fort)'])

print(df_mase.groupby('volume_quartile')['mase'].median())
print(df_mase.groupby('volume_quartile').apply(lambda g: (g['mase'] < 1).mean()))

volume_quartile
Q1 (faible)    2.581907
Q2             1.443258
Q3             1.056204
Q4 (fort)      0.851657
Name: mase, dtype: float64
volume_quartile
Q1 (faible)    0.087919
Q2             0.214079
Q3             0.438385
Q4 (fort)      0.666229
dtype: float64


In [21]:
df_mase_biais = test_fe.groupby(['item_id', 'store_id']).apply(
    lambda g: (g['y_pred'] - g['quantite']).mean()
).rename('biais')

df_mase_full = df_mase.join(df_mase_biais)

print(df_mase_full.groupby('volume_quartile')['biais'].mean())

volume_quartile
Q1 (faible)    0.079371
Q2             0.053355
Q3             0.008188
Q4 (fort)     -0.119103
Name: biais, dtype: float64


In [22]:
model_quantile = lgb.LGBMRegressor(
    n_estimators=100,
    random_state=42,
    objective='quantile',
    alpha=0.6  # 0.5 = symétrique (défaut), plus proche de 1 = pénalise plus la sous-prédiction
)

model_quantile.fit(X_train, y_train, categorical_feature=cat_features)

y_pred_quantile = model_quantile.predict(X_test).clip(min=0)

wape_quantile = wape(y_test, pd.Series(y_pred_quantile, index=y_test.index))
print(f"WAPE (quantile alpha=0.6) : {wape_quantile:.4f}")
print(f"WAPE (baseline symétrique) : 0.7228")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.204575 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9
WAPE (quantile alpha=0.6) : 0.6761
WAPE (baseline symétrique) : 0.7228


In [23]:
test_fe['y_pred_quantile'] = y_pred_quantile

biais_quantile = test_fe.groupby(['item_id', 'store_id']).apply(
    lambda g: (g['y_pred_quantile'] - g['quantite']).mean()
).rename('biais_quantile')

df_biais_quantile = df_mase[['volume_quartile']].join(biais_quantile)
print(df_biais_quantile.groupby('volume_quartile')['biais_quantile'].mean())

volume_quartile
Q1 (faible)   -0.114668
Q2            -0.129017
Q3            -0.099319
Q4 (fort)     -0.148248
Name: biais_quantile, dtype: float64


In [24]:
for alpha_test in [0.3, 0.5, 0.7, 0.8]:
    m = lgb.LGBMRegressor(n_estimators=100, random_state=42, objective='quantile', alpha=alpha_test)
    m.fit(X_train, y_train, categorical_feature=cat_features)
    pred = m.predict(X_test).clip(min=0)
    biais_global = (pred - y_test).mean()
    wape_test = wape(y_test, pd.Series(pred, index=y_test.index))
    print(f"alpha={alpha_test} | biais global={biais_global:.4f} | WAPE={wape_test:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.792462 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9
alpha=0.3 | biais global=-0.8567 | WAPE=0.7194
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.961916 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9
alpha=0.5 | biais global=-0.4405 | WAPE=0.6563
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.045000 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can

In [25]:
model_65 = lgb.LGBMRegressor(n_estimators=100, random_state=42, objective='quantile', alpha=0.65)
model_65.fit(X_train, y_train, categorical_feature=cat_features)
pred_65 = model_65.predict(X_test).clip(min=0)

wape_65 = wape(y_test, pd.Series(pred_65, index=y_test.index))
print(f"WAPE alpha=0.65 : {wape_65:.4f}")

test_fe['y_pred_65'] = pred_65
biais_65 = test_fe.groupby(['item_id', 'store_id']).apply(
    lambda g: (g['y_pred_65'] - g['quantite']).mean()
).rename('biais_65')

df_biais_65 = df_mase[['volume_quartile']].join(biais_65)
print(df_biais_65.groupby('volume_quartile')['biais_65'].mean())

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.676154 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9
WAPE alpha=0.65 : 0.7076
volume_quartile
Q1 (faible)   -0.038301
Q2            -0.002388
Q3             0.076035
Q4 (fort)      0.210292
Name: biais_65, dtype: float64
